# 03 Silver Layer Cleaning

This notebook transforms validated bronze tables into the silver layer.

Silver-layer responsibilities:
- Keep the original healthcare meaning of each table
- Standardize column names
- Trim string values
- Cast dates, timestamps, and numeric columns
- Add stable hash keys using `source_state`
- Deduplicate records using table-specific business keys
- Save cleaned Delta tables to `healthcare_demo.healthcare_silver`

Important design rule: because the project combines 5 states, every key must include `source_state`.

## Imports

In [0]:
# Spark column / type / window APIs used throughout the notebook
from pyspark.sql.functions import (
    col, current_timestamp, trim, to_date, to_timestamp,
    sha2, concat_ws, lit, regexp_replace, year,
    row_number, lower, upper, initcap, when
)
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType
from pyspark.sql import Row
from pyspark.sql.window import Window

# Shared validation helper - records data quality checks to _metadata tables
import sys
sys.path.append('/Workspace/Shared/healthcare_demo')
from validation_helper import ValidationRun

## 0. Validation Run Setup

This notebook persists data quality checks to `_metadata.validation_runs` and
`_metadata.validation_results`. Error-level failures will block downstream
notebooks.

In [0]:
run = ValidationRun(
    spark=spark,
    catalog="healthcare_demo",
    notebook_name="03_silver_cleaning",
    layer="silver"
)

print(f"Validation run started: {run.run_id}")

Validation run started: a29d38e1-75a6-45aa-905a-b56015a52b30


## 1. Configuration

In [0]:
catalog_name = "healthcare_demo"
bronze_schema = "healthcare_bronze"
silver_schema = "healthcare_silver"

# Analytical scope for this BI/ML project
# Bronze keeps the full raw Synthea history.
# Silver keeps only encounter-based activity from 2021 to 2025.
analysis_start_year = 2021
analysis_end_year = 2025

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")


DataFrame[]

## 2. Source Tables

In [0]:
# Master list — drives the main pipeline loop.
tables = [
    # Core dimensions / entities first
    "patients",
    "encounters",
    "organizations",
    "providers",
    "payers",

    # Encounter-linked clinical and financial tables
    "claims",
    "conditions",
    "observations",
    "medications",
    "procedures",
    "immunizations",
    "allergies",
    "careplans",
    "devices",
    "supplies",

    # Non-encounter history table
    "payer_transitions"
]

# Tables whose encounter represents WHEN the event happened.
# These are scoped by the encounter window (inner join to silver.encounters).
encounter_event_tables = [
    "observations",
    "procedures",
    "medications",     # treated as event for now; revisit separately
    "immunizations",
    "supplies"
]

# Tables that represent longitudinal clinical STATE.
# These are scoped by interval overlap with the analysis window,
# regardless of when the originating encounter occurred.
state_tables = [
    "conditions",
    "allergies",
    "careplans",
    "devices"
]

# Combined list — kept for backward compatibility with the rest of the notebook.
encounter_linked_tables = encounter_event_tables + state_tables

# Claims still links via appointmentid.
claims_encounter_fk = "appointmentid"

## 3. Silver Key Definitions

These keys are used for deduplication and hash-key generation.

All keys include `source_state` because the project combines files from multiple states.

In [0]:
deduplication_keys = {
    # Core entities
    "patients": ["source_state", "id"],
    "encounters": ["source_state", "id"],
    "organizations": ["source_state", "id"],
    "providers": ["source_state", "id"],
    "payers": ["source_state", "id"],

    # Financial
    "claims": ["source_state", "id"],

    # Clinical event tables
    "conditions": ["source_state", "patient", "encounter", "code"],
    "observations": ["source_state", "patient", "encounter", "code", "date", "description", "value"],
    "medications": ["source_state", "start", "stop", "patient", "encounter", "code", "description", "reasoncode"],
    "procedures": ["source_state", "start", "patient", "encounter", "code"],
    "immunizations": ["source_state", "date", "patient", "encounter", "code"],
    "allergies": ["source_state", "patient", "encounter", "code"],
    "careplans": ["source_state", "id"],

    # Additional healthcare tables
    "devices": ["source_state", "start", "patient", "encounter", "code"],
    "supplies": ["source_state", "date", "patient", "encounter", "code", "description", "quantity"],
    "payer_transitions": ["source_state", "patient", "start_date", "end_date", "payer"]
}

## 4. Date, Timestamp, and Numeric Column Rules

In [0]:
date_columns_by_table = {
    "patients": ["birthdate", "deathdate"],
    "payer_transitions": ["start_date", "end_date"],
    "observations": ["date"],
    "immunizations": ["date"],
    "supplies": ["date"]
}

timestamp_columns_by_table = {
    "encounters": ["start", "stop"],
    "conditions": ["start", "stop"],
    "procedures": ["start", "stop"],
    "medications": ["start", "stop"],
    "allergies": ["start", "stop"],
    "careplans": ["start", "stop"],
    "devices": ["start", "stop"],
}

numeric_columns_by_table = {
    "patients": ["healthcare_expenses", "healthcare_coverage", "income"],
    "encounters": ["base_encounter_cost", "total_claim_cost", "payer_coverage"],
    "procedures": ["base_cost"],
    "medications": ["base_cost", "payer_coverage", "dispenses", "totalcost"],
    "immunizations": ["base_cost"],
    "claims": ["outstanding1", "outstanding2", "outstandingp"],
    "supplies": ["quantity"]
}

## 5. Helper Functions

Generic cleaning helpers come first (column names, trimming, type casting),
followed by dedup, scope-filter, and domain-specific cleanup helpers.

In [0]:
def standardize_column_names(df):
    """
    Converts column names to lowercase and replaces spaces/hyphens with underscores.
    """
    for old_col in df.columns:
        new_col = (
            old_col.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        df = df.withColumnRenamed(old_col, new_col)
    return df


def trim_string_columns(df):
    """
    Removes leading and trailing spaces from string columns.
    """
    string_cols = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
    ]

    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))

    return df


def cast_date_columns(df, table_name):
    """
    Casts known date columns to DateType.
    """
    for c in date_columns_by_table.get(table_name, []):
        if c in df.columns:
            df = df.withColumn(c, to_date(col(c)))
    return df


def cast_timestamp_columns(df, table_name):
    """
    Casts known timestamp columns to TimestampType.
    """
    for c in timestamp_columns_by_table.get(table_name, []):
        if c in df.columns:
            df = df.withColumn(c, to_timestamp(col(c)))
    return df


def cast_numeric_columns(df, table_name):
    """
    Casts known numeric columns to DoubleType after removing dollar signs and commas if present.
    """
    for c in numeric_columns_by_table.get(table_name, []):
        if c in df.columns:
            df = df.withColumn(
                c,
                regexp_replace(col(c).cast("string"), "[$,]", "").cast(DoubleType())
            )
    return df


def add_hash_key(df, key_name, key_columns):
    """
    Adds a stable SHA-256 hash key based on selected columns.
    Nulls are converted to a placeholder so the hash remains deterministic.
    """
    existing_key_columns = [c for c in key_columns if c in df.columns]

    return df.withColumn(
        key_name,
        sha2(
            concat_ws(
                "||",
                *[
                    col(c).cast("string")
                    for c in existing_key_columns
                ]
            ),
            256
        )
    )


def deduplicate_by_key(df, key_columns):
    """
    Keeps one row per key, deterministically.

    When rows share key columns but differ in other columns, the survivor is
    chosen by ascending lexicographic ordering of all non-key columns (nulls last).
    This guarantees the same rows survive on every re-run.
    """
    existing_key_columns = [c for c in key_columns if c in df.columns]

    if not existing_key_columns:
        return df.dropDuplicates()

    non_key_columns = [c for c in df.columns if c not in existing_key_columns]

    if not non_key_columns:
        # Nothing to break ties on — duplicates must be bit-identical
        return df.dropDuplicates(existing_key_columns)

    window_spec = Window.partitionBy(*existing_key_columns).orderBy(
        *[col(c).asc_nulls_last() for c in non_key_columns]
    )

    return (
        df.withColumn("_dedup_row_num", row_number().over(window_spec))
          .filter(col("_dedup_row_num") == 1)
          .drop("_dedup_row_num")
    )

def add_silver_metadata(df):
    return df.withColumn("silver_processed_at", current_timestamp())

def filter_encounters_to_analysis_window(df):
    """
    Keeps only encounters whose start timestamp falls within the project analysis window.
    """
    return df.filter(year(col("start")).between(analysis_start_year, analysis_end_year))


def filter_to_valid_encounters(df, table_name, valid_encounters_df):
    """
    Keeps only rows connected to encounters already retained in silver.encounters.
    This prevents old lifetime-history records from leaking into BI or ML tables.
    """
    if valid_encounters_df is None:
        raise ValueError("valid_encounters_df is not available. Process encounters before encounter-linked tables.")

    if table_name == "claims":
        fk_col = claims_encounter_fk
    elif table_name in encounter_linked_tables:
        fk_col = "encounter"
    else:
        return df

    if fk_col not in df.columns:
        return df

    return (
        df
        .join(
            valid_encounters_df,
            (df["source_state"] == valid_encounters_df["valid_source_state"]) &
            (df[fk_col] == valid_encounters_df["valid_encounter_id"]),
            "inner"
        )
        .drop("valid_source_state", "valid_encounter_id")
    )
def filter_state_to_analysis_window(df):
    """
    Keeps state rows that were active at any point during the analysis window.
    Active means: start <= window_end AND (stop is null OR stop >= window_start).
    Used for state tables (conditions, allergies, careplans, devices) where the
    originating encounter may pre-date the window but the clinical state persists.
    """
    window_start_ts = to_timestamp(lit(f"{analysis_start_year}-01-01 00:00:00"))
    window_end_ts   = to_timestamp(lit(f"{analysis_end_year}-12-31 23:59:59"))

    return df.filter(
        (col("start") <= window_end_ts) &
        (col("stop").isNull() | (col("stop") >= window_start_ts))
    )

# Acronyms that should remain uppercase, mapped from their lowercase form
encounter_class_overrides = {
    "snf": "SNF",
    "urgentcare": "Urgent Care",
    "er": "ER",
    "icu": "ICU",
}

def standardize_encounter_class(df, column_name="encounterclass"):
    """
    Standardizes encounter class strings:
    - Title-cases each value (Ambulatory, Wellness, ...)
    - Preserves known acronyms in UPPERCASE (SNF, ER, ICU, ...)
    Idempotent — safe to run on already-standardized data.
    """
    if column_name not in df.columns:
        return df

    # Start with title case
    result = df.withColumn(
        column_name,
        initcap(col(column_name))
    )

    # Override known acronyms
    for raw, display in encounter_class_overrides.items():
        result = result.withColumn(
            column_name,
            when(upper(col(column_name)) == raw.upper(), display)
            .otherwise(col(column_name))
        )

    return result

def strip_snomed_suffix(df, column_name):
    
    """Removes trailing SNOMED axis qualifiers from a description column.
    
    Examples:
        'Hypertension (disorder)'        -> 'Hypertension'
        'Appendectomy (procedure)'       -> 'Appendectomy'
        'Chemotherapy (regime/therapy)'  -> 'Chemotherapy'
        'School (environment)'           -> 'School'
        'Body weight (observable entity)'-> 'Body weight'
    
    Only strips the recognized SNOMED axis labels. Other parentheticals
    (e.g. 'Vaccination (influenza)') are preserved.
    """
    if column_name not in df.columns:
        return df

    snomed_qualifiers = (
        r"procedure|regime/therapy|disorder|environment|"
        r"finding|observable entity|situation|qualifier value|"
        r"substance|product|morph abnormality|body structure|"
        r"event|occupation|person"
    )
    pattern = rf"\s*\(({snomed_qualifiers})\)\s*$"

    return df.withColumn(
        column_name,
        trim(regexp_replace(col(column_name), pattern, ""))
    )

def titlecase_column(df, column_name):
    """
    Converts the first letter of each word to uppercase, rest lowercase.
    
    Examples:
        'white'              -> 'White'
        'black'              -> 'Black'
        'american indian'    -> 'American Indian'
        'NATIVE'             -> 'Native'
        'hispanic'           -> 'Hispanic'
    
    Null values are preserved as null.
    """
    if column_name not in df.columns:
        return df
    
    return df.withColumn(column_name, initcap(col(column_name)))

## 6. Relationship Hash Keys

These keys make the silver layer relationship-ready without joining tables yet.

In [0]:
def add_relationship_keys(df, table_name):
    """
    Adds relationship keys where source columns exist.
    These keys are designed for clean joins in the gold layer.
    """

    # Primary table/event key
    if table_name in deduplication_keys:
        df = add_hash_key(df, f"{table_name[:-1] if table_name.endswith('s') else table_name}_key", deduplication_keys[table_name])

    # Common relationship keys
    if "patient" in df.columns:
        df = add_hash_key(df, "patient_key", ["source_state", "patient"])

    if "patientid" in df.columns:
        df = add_hash_key(df, "patient_key", ["source_state", "patientid"])

    if "encounter" in df.columns:
        df = add_hash_key(df, "encounter_key", ["source_state", "encounter"])

    if table_name == "encounters" and "id" in df.columns:
        df = add_hash_key(df, "encounter_key", ["source_state", "id"])

    if "provider" in df.columns:
        df = add_hash_key(df, "provider_key", ["source_state", "provider"])

    if "providerid" in df.columns:
        df = add_hash_key(df, "provider_key", ["source_state", "providerid"])

    if "organization" in df.columns:
        df = add_hash_key(df, "organization_key", ["source_state", "organization"])

    if "payer" in df.columns:
        df = add_hash_key(df, "payer_key", ["source_state", "payer"])

    if "claimid" in df.columns:
        df = add_hash_key(df, "claim_key", ["source_state", "claimid"])

    if table_name == "claims" and "id" in df.columns:
        df = add_hash_key(df, "claim_key", ["source_state", "id"])

    if "appointmentid" in df.columns:
        df = add_hash_key(df, "appointment_encounter_key", ["source_state", "appointmentid"])

    return df

## 7. Silver Cleaning Pipeline

For each bronze table:
1. Standardize column names and trim string values
2. Cast date / timestamp / numeric columns
3. Deduplicate using the per-table business key
4. Apply table-specific cleanup (encounter class, SNOMED suffix, title-casing)
5. Apply the analysis-window scope (encounter window or state interval overlap)
6. Add relationship hash keys and a silver-processed timestamp
7. Write to Delta and record a summary row for the validation log

In [0]:
silver_summary = []
valid_encounters_df = None

for table in tables:

    bronze_table = f"{catalog_name}.{bronze_schema}.{table}"
    silver_table = f"{catalog_name}.{silver_schema}.{table}"

    df = spark.table(bronze_table)
    bronze_count = df.count()

    df_clean = (
        df
        .transform(standardize_column_names)
        .transform(trim_string_columns)
    )

    df_clean = cast_date_columns(df_clean, table)
    df_clean = cast_timestamp_columns(df_clean, table)
    df_clean = cast_numeric_columns(df_clean, table)

    duplicate_key = deduplication_keys.get(table, [])
    before_dedup_count = df_clean.count()
    df_clean = deduplicate_by_key(df_clean, duplicate_key)
    after_dedup_count = df_clean.count()

    # Encounter-specific cleanup
    if table == "encounters":
        df_clean = standardize_encounter_class(df_clean, "encounterclass")
        df_clean = strip_snomed_suffix(df_clean, "reasondescription")

    # Patient-specific cleanup
    if table == "patients":
        df_clean = titlecase_column(df_clean, "race")
        df_clean = titlecase_column(df_clean, "ethnicity")

    # Universal cleanup — strip SNOMED axis suffixes from any description column.
    # The helper is a no-op for tables that don't have a description column.
    df_clean = strip_snomed_suffix(df_clean, "description")

    # Apply the project analytical scope in Silver.
    # 1) encounters: keep only 2021-2025 encounters.
    # 2) encounter-linked tables: keep only rows connected to retained encounters.
    before_scope_count = df_clean.count()

    if table == "encounters":
        df_clean = filter_encounters_to_analysis_window(df_clean)
        valid_encounters_df = (
            df_clean
            .select(
                col("source_state").alias("valid_source_state"),
                col("id").alias("valid_encounter_id")
            )
            .distinct()
        )
    elif table in state_tables:
        # Longitudinal state — use interval overlap, not encounter membership
        df_clean = filter_state_to_analysis_window(df_clean)
    elif table in encounter_event_tables or table == "claims":
        # Transactional events — must tie to an in-scope encounter
        df_clean = filter_to_valid_encounters(df_clean, table, valid_encounters_df)

    after_scope_count = df_clean.count()

    df_clean = add_relationship_keys(df_clean, table)
    df_clean = add_silver_metadata(df_clean)

    (
        df_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )

    if table == "encounters":
        window_label = f"{analysis_start_year}-{analysis_end_year} (encounter scope)"
    elif table in state_tables:
        window_label = f"{analysis_start_year}-{analysis_end_year} (state interval overlap)"
    elif table in encounter_event_tables or table ==   "claims":
        window_label = f"{analysis_start_year}-{analysis_end_year} (encounter scope)"
    else:
        window_label = "Not encounter-scoped"

    silver_summary.append(
        Row(
            table_name=table,
            bronze_rows=bronze_count,
            silver_rows=after_scope_count,
            duplicates_removed=before_dedup_count - after_dedup_count,
            rows_removed_by_analysis_scope=before_scope_count - after_scope_count,
            analysis_window=window_label,
            deduplication_key=", ".join(duplicate_key)
        )
    )

silver_summary_df = spark.createDataFrame(silver_summary)
display(silver_summary_df)

# Record per-table summary info checks
for row in silver_summary:
    table_name = row["table_name"]

    run.add_result(
        check_id=f"silver.{table_name}.silver_row_count",
        check_name=f"Silver row count for {table_name}",
        check_category="row_count",
        target_table=table_name,
        severity="info",
        expected=">0",
        actual=str(row["silver_rows"]),
        status="passed",
        message=f"Bronze→Silver: {row['bronze_rows']:,} → {row['silver_rows']:,}"
    )

    run.add_result(
        check_id=f"silver.{table_name}.duplicates_removed",
        check_name=f"Duplicates removed for {table_name}",
        check_category="dedup",
        target_table=table_name,
        severity="info",
        expected="N/A",
        actual=str(row["duplicates_removed"]),
        status="passed",
        message=f"Dedup key: {row['deduplication_key']}"
    )

    run.add_result(
        check_id=f"silver.{table_name}.scope_filter_removed",
        check_name=f"Rows removed by analysis scope for {table_name}",
        check_category="scope_filter",
        target_table=table_name,
        severity="info",
        expected="N/A",
        actual=str(row["rows_removed_by_analysis_scope"]),
        status="passed",
        message=row["analysis_window"]
    )

print(f"Recorded {len(silver_summary) * 3} pipeline summary checks.")


table_name,bronze_rows,silver_rows,duplicates_removed,rows_removed_by_analysis_scope,analysis_window,deduplication_key
patients,22817,22817,0,0,Not encounter-scoped,"source_state, id"
encounters,827172,459785,0,367387,2021-2025 (encounter scope),"source_state, id"
organizations,15449,15449,0,0,Not encounter-scoped,"source_state, id"
providers,15449,15449,0,0,Not encounter-scoped,"source_state, id"
payers,50,50,0,0,Not encounter-scoped,"source_state, id"
claims,1539911,850177,0,689734,2021-2025 (encounter scope),"source_state, id"
conditions,528897,448416,0,80481,2021-2025 (state interval overlap),"source_state, patient, encounter, code"
observations,9460263,6071198,16288,3372777,2021-2025 (encounter scope),"source_state, patient, encounter, code, date, description, value"
medications,712739,390392,0,322347,2021-2025 (encounter scope),"source_state, start, stop, patient, encounter, code, description, reasoncode"
procedures,2029842,1417608,0,612234,2021-2025 (encounter scope),"source_state, start, patient, encounter, code"


Recorded 48 pipeline summary checks.


## 8. Silver Row Count Validation

In [0]:
silver_counts = []

for table in tables:
    df = spark.table(f"{catalog_name}.{silver_schema}.{table}")
    silver_counts.append((table, df.count()))

silver_counts_df = spark.createDataFrame(
    silver_counts,
    ["table_name", "row_count"]
)

display(silver_counts_df)

# Record row-count assertions — every silver table must be non-empty
for row in silver_counts_df.collect():
    table_name = row["table_name"]
    row_count = row["row_count"]
    is_passing = (row_count > 0)

    run.add_result(
        check_id=f"silver.{table_name}.non_empty",
        check_name=f"Silver table {table_name} must be non-empty",
        check_category="row_count",
        target_table=table_name,
        severity="error",
        expected=">0",
        actual=str(row_count),
        status="passed" if is_passing else "failed",
        message="" if is_passing else "Silver table is empty — silver pipeline broken"
    )

print(f"Recorded {silver_counts_df.count()} non-empty assertions.")

table_name,row_count
patients,22817
encounters,459785
organizations,15449
providers,15449
payers,50
claims,850177
conditions,448416
observations,6071198
medications,390392
procedures,1417608


Recorded 16 non-empty assertions.


## 9. Silver Schema Review

In [0]:
for table in tables:
    print(f"\n===== {table.upper()} =====")
    spark.table(f"{catalog_name}.{silver_schema}.{table}").printSchema()


===== PATIENTS =====
root
 |-- id: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- deathdate: date (nullable = true)
 |-- ssn: string (nullable = true)
 |-- drivers: string (nullable = true)
 |-- passport: string (nullable = true)
 |-- prefix: string (nullable = true)
 |-- first: string (nullable = true)
 |-- middle: string (nullable = true)
 |-- last: string (nullable = true)
 |-- suffix: string (nullable = true)
 |-- maiden: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthplace: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- fips: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- healthcare_expenses: double (n

## 10. Silver Relationship Checks

These checks do not remove records. They show whether silver tables are relationship-ready.

In [0]:
def check_fk(child_table, child_fk_col, parent_table, parent_pk_col, soft=False):
    child_df = spark.table(f"{catalog_name}.{silver_schema}.{child_table}")
    parent_df = spark.table(f"{catalog_name}.{silver_schema}.{parent_table}")

    if child_fk_col not in child_df.columns or parent_pk_col not in parent_df.columns:
        return Row(
            child_table=child_table, child_fk=child_fk_col,
            parent_table=parent_table, parent_pk=parent_pk_col,
            orphan_count=None, note="Column missing"
        )

    orphan_count = (
        child_df
        .filter(col(child_fk_col).isNotNull())
        .join(
            parent_df.select(
                col("source_state").alias("parent_source_state"),
                col(parent_pk_col).alias("parent_id")
            ).dropDuplicates(),
            (child_df["source_state"] == col("parent_source_state")) &
            (child_df[child_fk_col] == col("parent_id")),
            "left_anti"
        )
        .count()
    )

    if soft:
        note = f"Soft FK (state table) — {orphan_count} rows reference encounters outside analysis window. Expected."
    else:
        note = "Checked"

    return Row(
        child_table=child_table, child_fk=child_fk_col,
        parent_table=parent_table, parent_pk=parent_pk_col,
        orphan_count=orphan_count, note=note
    )

fk_checks = []

# Patient relationships
for table in [
    "encounters", "conditions", "observations", "procedures", "medications",
    "immunizations", "allergies", "careplans", "devices",
    "supplies", "payer_transitions"
]:
    fk_checks.append(check_fk(table, "patient", "patients", "id"))

fk_checks.append(check_fk("claims", "patientid", "patients", "id"))

# Encounter relationships
for table in encounter_event_tables:
    fk_checks.append(check_fk(table, "encounter", "encounters", "id"))

for table in state_tables:
    fk_checks.append(check_fk(table, "encounter", "encounters", "id", soft=True))

fk_checks.append(check_fk("claims", "appointmentid", "encounters", "id"))

# Provider, organization, payer relationships
fk_checks.append(check_fk("encounters", "provider", "providers", "id"))
fk_checks.append(check_fk("encounters", "organization", "organizations", "id"))
fk_checks.append(check_fk("encounters", "payer", "payers", "id"))
fk_checks.append(check_fk("providers", "organization", "organizations", "id"))
fk_checks.append(check_fk("claims", "providerid", "providers", "id"))
fk_checks.append(check_fk("payer_transitions", "payer", "payers", "id"))

fk_checks_df = spark.createDataFrame(fk_checks)
display(fk_checks_df)

# Define which FK checks are "soft" (state-table encounter FKs — expected non-zero)
soft_fk_pairs = {
    ("conditions", "encounter"),
    ("allergies", "encounter"),
    ("careplans", "encounter"),
    ("devices", "encounter"),
}

for row in fk_checks_df.collect():
    child_table = row["child_table"]
    child_fk = row["child_fk"]
    parent_table = row["parent_table"]
    orphan_count = row["orphan_count"]
    note = row["note"]
    is_soft = (child_table, child_fk) in soft_fk_pairs

    if orphan_count is None:
        # Column missing - record as failed
        run.add_result(
            check_id=f"silver.{child_table}.fk_{parent_table}.{child_fk}",
            check_name=f"FK {child_table}.{child_fk} -> {parent_table}",
            check_category="fk_integrity",
            target_table=child_table,
            target_column=child_fk,
            severity="error",
            expected="column exists",
            actual="missing",
            status="failed",
            message=note
        )
        continue

    if is_soft:
        severity = "info"
        status = "passed"  # soft FKs are expected to have non-zero orphans
        msg = f"{orphan_count} rows reference encounters outside analysis window (expected for state table)"
    else:
        severity = "error"
        status = "passed" if orphan_count == 0 else "failed"
        msg = "" if orphan_count == 0 else f"{orphan_count} orphan FK references"

    run.add_result(
        check_id=f"silver.{child_table}.fk_{parent_table}.{child_fk}",
        check_name=f"FK {child_table}.{child_fk} -> {parent_table}.{row['parent_pk']}",
        check_category="fk_integrity",
        target_table=child_table,
        target_column=child_fk,
        severity=severity,
        expected="0" if not is_soft else ">=0 (soft FK)",
        actual=str(orphan_count),
        status=status,
        message=msg
    )

print(f"Recorded {fk_checks_df.count()} FK integrity checks.")


child_table,child_fk,parent_table,parent_pk,orphan_count,note
encounters,patient,patients,id,0,Checked
conditions,patient,patients,id,0,Checked
observations,patient,patients,id,0,Checked
procedures,patient,patients,id,0,Checked
medications,patient,patients,id,0,Checked
immunizations,patient,patients,id,0,Checked
allergies,patient,patients,id,0,Checked
careplans,patient,patients,id,0,Checked
devices,patient,patients,id,0,Checked
supplies,patient,patients,id,0,Checked


Recorded 28 FK integrity checks.


## 11. Date Sanity and Analysis Window Checks

Validates that:

- Every table with `start` and `stop` columns has `start <= stop` (where stop is not null)
- Every encounter in silver falls within the analysis window (2021-2025)

Hard errors. If any of these fail, downstream gold tables will have bad dates.

In [0]:
# Tables with start/stop pairs to check
tables_with_intervals = [
    "encounters",
    "conditions",
    "allergies",
    "careplans",
    "devices",
]

date_sanity_rows = []

for table in tables_with_intervals:
    df = spark.table(f"{catalog_name}.{silver_schema}.{table}")

    # stop may be null for ongoing state — only check rows where both are populated
    invalid_count = (
        df
        .filter(col("stop").isNotNull())
        .filter(col("start") > col("stop"))
        .count()
    )

    date_sanity_rows.append((table, invalid_count))

    run.add_result(
        check_id=f"silver.{table}.date_order",
        check_name=f"Date order: {table}.start <= {table}.stop",
        check_category="date_sanity",
        target_table=table,
        target_column="start, stop",
        severity="error",
        expected="0",
        actual=str(invalid_count),
        status="passed" if invalid_count == 0 else "failed",
        message="" if invalid_count == 0 else f"{invalid_count} rows where start > stop"
    )

date_sanity_df = spark.createDataFrame(date_sanity_rows, ["table_name", "invalid_date_order_count"])
display(date_sanity_df)
print(f"Recorded {len(tables_with_intervals)} date-order checks.")

table_name,invalid_date_order_count
encounters,0
conditions,0
allergies,0
careplans,0
devices,0


Recorded 5 date-order checks.


In [0]:
# Confirm every encounter falls within the analysis window
window_start = to_timestamp(lit(f"{analysis_start_year}-01-01 00:00:00"))
window_end   = to_timestamp(lit(f"{analysis_end_year}-12-31 23:59:59"))

encounters_df = spark.table(f"{catalog_name}.{silver_schema}.encounters")

out_of_window_count = (
    encounters_df
    .filter((col("start") < window_start) | (col("start") > window_end))
    .count()
)

run.add_result(
    check_id="silver.encounters.analysis_window",
    check_name=f"Encounters must fall within {analysis_start_year}-{analysis_end_year}",
    check_category="scope_filter",
    target_table="encounters",
    target_column="start",
    severity="error",
    expected="0",
    actual=str(out_of_window_count),
    status="passed" if out_of_window_count == 0 else "failed",
    message="" if out_of_window_count == 0 else f"{out_of_window_count} encounters outside analysis window"
)

print(f"Recorded analysis window check: {out_of_window_count} out-of-window encounters.")

Recorded analysis window check: 0 out-of-window encounters.


## 12. Silver Design Notes

- Bronze tables remain raw and unchanged.
- Silver tables keep the same table names to make the pipeline easy to trace.
- The analysis window is 2021–2025.
- Two scoping regimes apply to clinical tables:
  - Encounter-scoped (event tables): observations, procedures, medications,
    immunizations, supplies, and claims. Filtered by inner join to silver.encounters.
  - Interval-overlap (state tables): conditions, allergies, careplans, devices.
    Filtered by `start <= window_end AND (stop is null OR stop >= window_start)`.
    The originating encounter FK is preserved but is a soft reference — it may
    point to an encounter outside the analysis window.
- Relationship hash keys are added to support clean joins in gold.
- No analytics-level aggregations are performed in silver.
- Patient FKs remain strict across all tables.
- Gold layer can later rename these into `dim_` and `fact_` tables if needed.

## 13. Persist Results and Assert

Write all check results to `_metadata.validation_results` and assert no
error-level failures.

In [0]:
summary = run.commit()

print(f"Run ID:         {summary['run_id']}")
print(f"Status:         {summary['status']}")
print(f"Total checks:   {summary['total']}")
print(f"  Passed:       {summary['passed']}")
print(f"  Warned:       {summary['warned']}")
print(f"  Failed:       {summary['failed']}")
print(f"Duration:       {summary['duration_seconds']:.2f}s")

Run ID:         a29d38e1-75a6-45aa-905a-b56015a52b30
Status:         passed
Total checks:   98
  Passed:       98
  Warned:       0
  Failed:       0
Duration:       253.80s


In [0]:
run.assert_no_failures()